# PT-W4-D5 实验 · Agent Mapping：数字员工是语义模型的组合层

**概念问题**：前四层（Ontology / Lifecycle / Rule / Capability+Policy）描述世界，Agent Mapping 不新增业务语义，只声明**谁、在什么边界内、带着什么授权**消费这些语义。

本 notebook 做四件事：
1. 把 D1-D4 的前四层成果写成可运行的精简结构
2. 定义三张 Agent Mapping Card（招商 / 运营 / 财务）
3. 跑**委托边界校验器**（含 Membership=excluded 反例拦截）——"定义即约束，约束即可验证"
4. 跑 A101 验证场景：问题路由（owner/subscriber）+ 运营数字员工的 L1→L5 全链回答

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 1. 前四层语义模型（精简版，D1-D4 成果）

- **Capability（D4）**：`delegation` 字段 = 委托边界③——这个能力允不允许交给数字员工。`MEMBERSHIP` 对应 MI 追溯矩阵的 `excluded`（"Do not migrate member cards, points…"），`delegation=False` 就是硬闸门。
- **Lifecycle（D2）**：A101 关联租约的状态机节点。
- **Rule（D3）**：Guard 规则卡——主语是"世界状态"，不是人。
- **Policy①（D4）**：审批路径——主语是"人/角色"。

In [ ]:
# ── Capability：能力卡（D4）。delegation=False = 委托边界③不允许交给数字员工
CAPABILITIES = {
    "CRE-OPS-012": dict(name="创建巡检任务", context="03 租赁管理", maturity="accepted",
                        delegation=True, effect_policy="conditional_write", review_gate=True),
    "CRE-CON-024a": dict(name="终止合同执行", context="02 合同管理", maturity="accepted",
                         delegation=True, effect_policy="conditional_write", review_gate=True),
    "CRE-FIN-021": dict(name="费用减免起草（仅草稿态）", context="04 财务管理", maturity="accepted",
                        delegation=True, effect_policy="conditional_write", review_gate=True),
    "MEMBERSHIP": dict(name="会员卡/积分/礼券", context="07 会员管理", maturity="excluded",
                       delegation=False, effect_policy=None, review_gate=None),
}

# ── Lifecycle（D2）：A101 关联租约当前状态
LEASE_STATE = {
    "A101": dict(lease="L-2025-088", state="TerminatedPendingInspection",
                 inspection="未完成", occupancy="占用中"),
}

# ── Rule（D3）：Guard —— 世界状态是否满足
GUARDS = {
    "CRE-R-003": dict(text="存在未完成退租流程的 Space 不可出租",
                      check=lambda s: s["inspection"] == "已完成"),
}

# ── Policy①（D4）：审批路径 —— 社会授权是否完成
APPROVAL = {
    "终止申请": "K2 审批（已通过）",
    "创建巡检任务": "免审批（常规任务）",
    "减免单生效": "K2 审批（未发起）",
}
print(f"能力卡 {len(CAPABILITIES)} 张 | Guard {len(GUARDS)} 条 | A101 租约状态: {LEASE_STATE['A101']['state']}")

## 2. Agent Mapping Card：身份公式

`数字员工身份 = 岗位锚点 × 认知边界(Context×姿态) × 技能组合 × 授权配置 × 状态诚实`

- 岗位锚点挂**业务岗位轴**（7 岗位），不是 Skill 轴（README §5.4 双轴规则）
- 认知边界不是 Context 名单，是 **Context × 读写姿态** 矩阵
- 状态诚实（§7.2）：Skill 全是（候选）→ 员工只能是（规划）

In [ ]:
from dataclasses import dataclass

@dataclass
class AgentCard:
    id: str
    role_anchor: str          # 岗位锚点（业务岗位轴）
    contexts: dict            # 认知边界：context -> "RW"(读写) / "RO"(只读)；不列 = 不可见
    skills: list              # 技能组合：[{name, status, deps:[capability id]}]
    status: str = "规划"       # 状态诚实

de_sal = AgentCard("DE-SAL-01", "招商",
    {"01 招商管理": "RW", "03 租赁管理": "RO", "02 合同管理": "RO"},
    [dict(name="招商研究", status="候选·产品已列", deps=["CRE-SAL-001"])])  # SAL 域能力未登记 → 三角未闭合告警

de_ops = AgentCard("DE-OPS-01", "营运",
    {"05 运营管理": "RW", "03 租赁管理": "RW", "02 合同管理": "RO", "06 商户管理": "RO"},
    [dict(name="营运分析", status="候选·产品已列", deps=["CRE-OPS-012"]),
     dict(name="巡检创建", status="候选", deps=["CRE-OPS-012"])])

de_fin = AgentCard("DE-FIN-01", "财务",
    {"04 财务管理": "RW", "02 合同管理": "RO", "03 租赁管理": "RO", "05 运营管理": "RO"},
    [dict(name="AI 欠费分析", status="候选·产品已列", deps=["CRE-FIN-030"]),
     dict(name="减免单起草", status="候选", deps=["CRE-FIN-021"])])

REAL_CARDS = [de_sal, de_ops, de_fin]
for c in REAL_CARDS:
    print(f"{c.id}（{c.role_anchor}｜{c.status}）边界: {c.contexts}")

## 3. 委托边界校验器：定义即约束，约束即可验证

三条校验（对照 md 里的三条可执行校验）：
1. **③硬闸门**：依赖 `excluded` 能力 → FAIL（Membership 整域被 MI 矩阵排除，不得进任何数字员工清单）
2. **认知边界**：skill 依赖的能力所属 Context 必须在卡片边界内（不可见 = FAIL）
3. **状态诚实**：Skill 全候选 → 员工只能（规划）
4. 未知能力 ID → WARN（BCM 候选未转正，三角未闭合，非错误）

In [ ]:
def validate_de(card):
    issues = []
    for skill in card.skills:
        for dep in skill["deps"]:
            cap = CAPABILITIES.get(dep)
            if cap is None:
                issues.append(("WARN", f"{skill['name']} 依赖 {dep} 未在三角登记（BCM 候选未转正）"))
                continue
            if cap["maturity"] == "excluded":
                issues.append(("FAIL", f"{skill['name']} 依赖被排除能力 {dep}（{cap['name']}）——③硬闸门拦截"))
            elif cap["context"] not in card.contexts:
                issues.append(("FAIL", f"{skill['name']} 依赖的 {dep} 属于「{cap['context']}」，不在 {card.id} 认知边界内"))
    if any("候选" in s["status"] for s in card.skills) and card.status != "规划":
        issues.append(("FAIL", "Skill 全为候选，员工却不是（规划）——违反状态诚实"))
    return issues

print("== 三张真实卡片 ==")
for c in REAL_CARDS:
    for lv, msg in validate_de(c):
        print(f"  [{lv}] {c.id}: {msg}")
    print(f"{c.id}: {'✅ 通过（无 FAIL）' if not any(l=='FAIL' for l,_ in validate_de(c)) else '❌ 有 FAIL'}")

print("\n== 反例：会员数字员工（引用 MEMBERSHIP）==")
de_mem_bad = AgentCard("DE-MEM-99", "企划",  # 岗位锚点也挂错（业务岗位轴无"会员"岗）
    {"07 会员管理": "RW"},
    [dict(name="会员积分运营", status="候选", deps=["MEMBERSHIP"])])
for lv, msg in validate_de(de_mem_bad):
    print(f"  [{lv}] {msg}")
print("→ 定义阶段即被拦截，无需等到运行时才失败。这就是 Agent Mapping 作为校验层的价值。")

## 4. A101 验证场景：问题路由 + 运营数字员工的 L1→L5 全链

**路由规则**：主实体 Space 归属 03（域边界表：资源状态生命周期归 03）→ 对 03 有 RW 的员工是 **owner**，RO 的是 **subscriber**。

In [ ]:
def route(entity_context, cards):
    owners = [c.id for c in cards if c.contexts.get(entity_context) == "RW"]
    subs   = [c.id for c in cards if c.contexts.get(entity_context) == "RO"]
    return owners, subs

owners, subs = route("03 租赁管理", REAL_CARDS)
print(f'问题：「A101 铺位为什么不能出租？」')
print(f"路由：主实体 Space → 03 租赁管理 → owner={owners}（RW）  subscriber={subs}（RO，可订阅结论）\n")

# ── owner（运营 DE）拿五层模型作答 ──
s, g, cap = LEASE_STATE["A101"], GUARDS["CRE-R-003"], CAPABILITIES["CRE-OPS-012"]
print(f"L1 查 Entity(D1)     ：Space A101，身份=楼→层→铺位，占用状态={s['occupancy']}")
print(f"L2 查 Lifecycle(D2)  ：关联租约 {s['lease']} 处于 {s['state']}（查验{s['inspection']}）")
print(f"L3 查 Rule(D3)       ：CRE-R-003「{g['text']}」→ 判定={'通过' if g['check'](s) else '未通过'}")
print(f"L4 查 Policy①(D4)    ：终止申请={APPROVAL['终止申请']}｜创建巡检={APPROVAL['创建巡检任务']}")
print(f"L5 查 Capability(D4) ：{cap['name']} = {cap['effect_policy']}，人审门={cap['review_gate']}")
print(f"\n结论：A101 不能出租 —— 存在未完成退租流程（CRE-R-003）。")
print(f"建议：由 DE-OPS-01 发起创建巡检任务（免审批，但 conditional_write 需人确认后执行）；")
print(f"      查验完成 → InspectionCompleted 事件 → occupancy-effect → A101 自动释放（D2 状态机迁移，非 Agent 动作）。")

## 5. 认知边界热力图：Context × 读写姿态

数字员工之间的差异不在"聪明程度"，而在**看得见什么、能写什么**——这正是 Domain Model 域边界在组织面上的投影。

In [ ]:
import numpy as np
ctx_labels = ["01 招商管理", "02 合同管理", "03 租赁管理", "04 财务管理", "05 运营管理", "06 商户管理"]
M = np.zeros((len(REAL_CARDS), len(ctx_labels)))
for i, c in enumerate(REAL_CARDS):
    for j, ctx in enumerate(ctx_labels):
        v = c.contexts.get(ctx)
        M[i, j] = {"RW": 2, "RO": 1}.get(v, 0)

fig, ax = plt.subplots(figsize=(8, 3.4))
im = ax.imshow(M, cmap="YlGn", vmin=0, vmax=2)
ax.set_xticks(range(len(ctx_labels)), ctx_labels, rotation=20, ha="right")
ax.set_yticks(range(len(REAL_CARDS)), [f"{c.id}\n{c.role_anchor}" for c in REAL_CARDS])
for i in range(len(REAL_CARDS)):
    for j in range(len(ctx_labels)):
        ax.text(j, i, ["不可见", "只读", "读写"][int(M[i, j])], ha="center", va="center", fontsize=9)
ax.set_title("数字员工认知边界：Context × 读写姿态（Agent Mapping 的核心字段）")
fig.colorbar(im, ticks=[0, 1, 2]).ax.set_yticklabels(["不可见", "只读", "读写"])
fig.tight_layout()
fig.savefig("d5_agent_boundary.png", dpi=120)
plt.show()
print("RW=可写（仍受人审门/审批约束）| RO=可消费不可写 | 不可见=认知边界外。")
print("招商研究/欠费分析依赖的能力未登记（WARN）——三角未闭合前，员工只能停留在（规划）。")

## 结论

- 数字员工**不新增业务语义**：五层模型回答 A101 的每一步（L1→L5）用的都是 D1-D4 的构件，Agent Card 只提供"谁在回答、凭什么"。
- **三条校验全部可执行**：excluded 硬闸门、认知边界覆盖、状态诚实——定义阶段拦截错误，而非运行时。
- 明天 D6：把 D1-D5 六个部分组装成《MI CRE Enterprise Semantic Model v0.1》完整文档 + 验证场景设计。